In [10]:
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

import matplotlib.pyplot as plt
from sklearn.tree import plot_tree
from sklearn.model_selection import train_test_split
from sksurv.ensemble import RandomSurvivalForest , GradientBoostingSurvivalAnalysis
from sksurv.linear_model import CoxPHSurvivalAnalysis
from sksurv.metrics import concordance_index_censored, concordance_index_ipcw , integrated_brier_score
from sklearn.impute import SimpleImputer
from sksurv.util import Surv

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor
import category_encoders as ce
import mygene
from sklearn.preprocessing import MultiLabelBinarizer

In [11]:
mol_train = pl.read_csv("../data/raw/X_train/molecular_train.csv")

mol_train

ID,CHR,START,END,REF,ALT,GENE,PROTEIN_CHANGE,EFFECT,VAF,DEPTH
str,str,f64,f64,str,str,str,str,str,f64,f64
"""P100000""","""11""",1.19149248e8,1.19149248e8,"""G""","""A""","""CBL""","""p.C419Y""","""non_synonymous_codon""",0.083,1308.0
"""P100000""","""5""",1.31822301e8,1.31822301e8,"""G""","""T""","""IRF1""","""p.Y164*""","""stop_gained""",0.022,532.0
"""P100000""","""3""",7.769406e7,7.769406e7,"""G""","""C""","""ROBO2""","""p.?""","""splice_site_variant""",0.41,876.0
"""P100000""","""4""",1.06164917e8,1.06164917e8,"""G""","""T""","""TET2""","""p.R1262L""","""non_synonymous_codon""",0.43,826.0
"""P100000""","""2""",2.5468147e7,2.5468163e7,"""ACGAAGAGGGGGTGTTC""","""A""","""DNMT3A""","""p.E505fs*141""","""frameshift_variant""",0.0898,942.0
…,…,…,…,…,…,…,…,…,…,…
"""P131472""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null
"""P131505""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null
"""P131816""",null,null,null,null,null,"""MLL""","""MLL_PTD""","""PTD""",null,null


In [7]:
mol_df = pl.read_csv("../data/processed/X_test/molecular_test_preprocess.csv")
cl_df = pl.read_csv("../data/processed/X_test/clinical_test_preprocess.csv")


In [9]:
mol_df.filter(pl.col("ID") == "KYW1")

ID,CHR,START,END,EFFECT_0,EFFECT_1,EFFECT_2,EFFECT_3,VAF,DEPTH,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,AA_TYPE_0,AA_TYPE_1,AA_TYPE_2,AA_TYPE_3,AA_TYPE_4,IS_PT_UKNOWN,is_frameshift,is_non_sens_mutation,IS_MISSENSE,REF_is_SNV_A,REF_is_SNV_C,REF_is_SNV_G,REF_is_SNV_T,ALT_is_SNV_A,ALT_is_SNV_C,ALT_is_SNV_G,ALT_is_SNV_T,complex_nucleotide_REF,complex_nucleotide_ALT,is_transition,is_transversion,is_indel
str,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""KYW1""",2.0,0.116384,0.116384,0,0,0,1,0.384,0.615141,0.352332,0.30303,0.012048,0,1,0,0,1,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,1,0
"""KYW1""",5.0,0.815836,0.815836,0,0,1,1,0.21,0.185493,0.134715,0.175084,0.481928,0,1,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1
"""KYW1""",1.0,0.736273,0.736273,0,0,1,0,0.027,0.446294,0.012953,0.016835,0.012048,0,1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0
"""KYW1""",7.0,0.313617,0.313617,0,1,1,1,0.3639,0.196591,0.147668,0.117845,0.108434,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [5]:
cl_df

ID,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,is_normal_cyto,is_a_Man,is_a_Female,is_deletion_anomaly,is_inversion_anomaly,is_added_anomaly,is_down_syndrome,is_monosomy,is_7_deleted,is_trisomy_8,iso_chromosome,is_derived_chromosome,is_lost_sex_chromosome,is_added_chromsome,is_deleted_chromsome,is_inserted_chromsome,is_translocated_anomaly,is_complex_karyo
str,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""KYW1""",1.0,0.329356,0.101295,0.0324,0.393519,0.15894,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
"""KYW2""",0.886076,0.30358,0.214197,0.0311,0.615741,0.10596,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1,1
"""KYW3""",1.0,1.0,1.0,0.93675,0.828704,0.082781,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0
"""KYW4""",1.0,0.529833,0.354663,0.93675,0.430556,0.145695,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
"""KYW5""",0.050633,0.115513,0.127478,0.03,0.486111,0.089404,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""KYW1189""",0.35443,0.105012,0.037047,0.026,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
"""KYW1190""",0.35443,0.105012,0.037047,0.026,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1
"""KYW1191""",0.35443,0.105012,0.037047,0.026,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1


In [39]:
cl_id = cl_df["ID"].to_numpy()
mol_id = mol_df["ID"].to_numpy()

for elem in cl_id:
    if elem not in mol_id:
        print(elem)

KYW16
KYW17
KYW23
KYW25
KYW26
KYW31
KYW37
KYW51
KYW57
KYW59
KYW60
KYW73
KYW86
KYW89
KYW94
KYW96
KYW147
KYW150
KYW153
KYW176
KYW205
KYW207
KYW212
KYW222
KYW234
KYW241
KYW242
KYW244
KYW247
KYW256
KYW275
KYW290
KYW295
KYW302
KYW311
KYW343
KYW349
KYW366
KYW399
KYW410
KYW438
KYW439
KYW466
KYW467
KYW468
KYW472
KYW480
KYW484
KYW488
KYW490
KYW492
KYW494
KYW497
KYW501
KYW502
KYW508
KYW509
KYW510
KYW513
KYW517
KYW518
KYW527
KYW528
KYW532
KYW533
KYW540
KYW548
KYW549
KYW551
KYW558
KYW563
KYW567
KYW568
KYW570
KYW571
KYW572
KYW573
KYW575
KYW582
KYW584
KYW590
KYW591
KYW592
KYW596
KYW603
KYW607
KYW614
KYW619
KYW620
KYW622
KYW623
KYW627
KYW646
KYW649
KYW655
KYW656
KYW660
KYW673
KYW688
KYW689
KYW692
KYW697
KYW702
KYW706
KYW713
KYW742
KYW765
KYW769
KYW780
KYW784
KYW785
KYW826
KYW831
KYW866
KYW891
KYW899
KYW910
KYW927
KYW970
KYW975
KYW993
KYW996
KYW1017
KYW1020
KYW1030
KYW1031
KYW1075
KYW1090
KYW1098
KYW1099
KYW1105
KYW1108
KYW1117
KYW1120
KYW1161
KYW1172
KYW1178
KYW1183
KYW1185


In [109]:
merge_df = cl_df.join(other=mol_df, on="ID" , how="left")

In [137]:
merge_df_2 = cl_df.join(other=mol_df, on="ID" , how="left")

In [139]:
merge_df_2.filter(pl.col("ID") == "KYW1193")

ID,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,is_normal_cyto,is_a_Man,is_a_Female,is_deletion_anomaly,is_inversion_anomaly,is_added_anomaly,is_down_syndrome,is_monosomy,is_7_deleted,is_trisomy_8,iso_chromosome,is_derived_chromosome,is_lost_sex_chromosome,is_added_chromsome,is_deleted_chromsome,is_inserted_chromsome,is_translocated_anomaly,is_complex_karyo,CHR,START,END,EFFECT_0,EFFECT_1,EFFECT_2,EFFECT_3,VAF,DEPTH,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,AA_TYPE_0,AA_TYPE_1,AA_TYPE_2,AA_TYPE_3,AA_TYPE_4,IS_PT_UKNOWN,is_frameshift,is_non_sens_mutation,IS_MISSENSE,REF_is_SNV_A,REF_is_SNV_C,REF_is_SNV_G,REF_is_SNV_T,ALT_is_SNV_A,ALT_is_SNV_C,ALT_is_SNV_G,ALT_is_SNV_T,complex_nucleotide_REF,complex_nucleotide_ALT,is_transition,is_transversion,is_indel
str,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""KYW1193""",0.35443,0.105012,0.037047,0.026,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,10.0,0.534481,0.534481,0,0,0,1,0.42,0.3044,0.020725,0.016835,0.0,0,1,1,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,1,0
"""KYW1193""",0.35443,0.105012,0.037047,0.026,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,17.0,0.03031,0.03031,0,0,0,1,0.713,0.214031,1.0,1.0,0.060241,0,1,1,0,1,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,1,0


In [110]:
merge_df.filter(pl.col("ID") == "KYW1183")

ID,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,is_normal_cyto,is_a_Man,is_a_Female,is_deletion_anomaly,is_inversion_anomaly,is_added_anomaly,is_down_syndrome,is_monosomy,is_7_deleted,is_trisomy_8,iso_chromosome,is_derived_chromosome,is_lost_sex_chromosome,is_added_chromsome,is_deleted_chromsome,is_inserted_chromsome,is_translocated_anomaly,is_complex_karyo,CHR,START,END,EFFECT_0,EFFECT_1,EFFECT_2,EFFECT_3,VAF,DEPTH,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,AA_TYPE_0,AA_TYPE_1,AA_TYPE_2,AA_TYPE_3,AA_TYPE_4,IS_PT_UKNOWN,is_frameshift,is_non_sens_mutation,IS_MISSENSE,REF_is_SNV_A,REF_is_SNV_C,REF_is_SNV_G,REF_is_SNV_T,ALT_is_SNV_A,ALT_is_SNV_C,ALT_is_SNV_G,ALT_is_SNV_T,complex_nucleotide_REF,complex_nucleotide_ALT,is_transition,is_transversion,is_indel
str,f64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,i64,i64,i64,i64,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
"""KYW1183""",0.35443,0.105012,0.037047,0.026,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,-1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [111]:
col_median = ["START" , "END" , "mutations_per_gene" , "mutations_per_gene_effect" , "mutations_per_region" , "VAF" , "DEPTH"]

merge_df = merge_df.with_columns([
    pl.col(c).fill_null(pl.col(c).median()) for c in col_median if c in merge_df.columns
])

In [113]:
col_common = [
    "CHR", "EFFECT_0", "EFFECT_1", "EFFECT_2", "EFFECT_3",
    "AA_TYPE_0", "AA_TYPE_1", "AA_TYPE_2", "AA_TYPE_3", "AA_TYPE_4",
    "IS_PT_UKNOWN", "is_frameshift", "is_non_sens_mutation", "IS_MISSENSE",
    "REF_is_SNV_A", "REF_is_SNV_C", "REF_is_SNV_G", "REF_is_SNV_T",
    "ALT_is_SNV_A", "ALT_is_SNV_C", "ALT_is_SNV_G", "ALT_is_SNV_T",
    "complex_nucleotide_REF", "complex_nucleotide_ALT",
    "is_transition", "is_transversion", "is_indel"
]

new_cols = [
    "is_a_Man", "is_a_Female", "is_deletion_anomaly", "is_inversion_anomaly", "is_added_anomaly",
    "is_down_syndrome", "is_monosomy", "is_7_deleted", "is_trisomy_8", "iso_chromosome",
    "is_derived_chromosome", "is_lost_sex_chromosome", "is_added_chromsome", "is_deleted_chromsome",
    "is_inserted_chromsome", "is_translocated_anomaly", "is_complex_karyo"
]

col_common +=new_cols

# Exemple sur un DataFrame `df`
# Appliquer fill_null (strategy = "backward") uniquement sur les colonnes de col_common
merge_df = merge_df.with_columns([
    pl.col(c).fill_null(strategy="forward") for c in col_common if c in merge_df.columns
])

In [114]:
merge_df.null_count()

ID,BM_BLAST,WBC,ANC,MONOCYTES,HB,PLT,is_normal_cyto,is_a_Man,is_a_Female,is_deletion_anomaly,is_inversion_anomaly,is_added_anomaly,is_down_syndrome,is_monosomy,is_7_deleted,is_trisomy_8,iso_chromosome,is_derived_chromosome,is_lost_sex_chromosome,is_added_chromsome,is_deleted_chromsome,is_inserted_chromsome,is_translocated_anomaly,is_complex_karyo,CHR,START,END,EFFECT_0,EFFECT_1,EFFECT_2,EFFECT_3,VAF,DEPTH,mutations_per_gene,mutations_per_gene_effect,mutations_per_region,AA_TYPE_0,AA_TYPE_1,AA_TYPE_2,AA_TYPE_3,AA_TYPE_4,IS_PT_UKNOWN,is_frameshift,is_non_sens_mutation,IS_MISSENSE,REF_is_SNV_A,REF_is_SNV_C,REF_is_SNV_G,REF_is_SNV_T,ALT_is_SNV_A,ALT_is_SNV_C,ALT_is_SNV_G,ALT_is_SNV_T,complex_nucleotide_REF,complex_nucleotide_ALT,is_transition,is_transversion,is_indel
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [115]:
drop_var = ['MONOCYTES', 'is_a_Man', 'is_deletion_anomaly', 'is_inversion_anomaly', 'is_added_anomaly', 'is_monosomy', 'is_7_deleted', 'iso_chromosome', 'is_inserted_chromsome', 'is_translocated_anomaly', 'START', 'END', 'EFFECT_0', 'EFFECT_1', 'EFFECT_2', 'EFFECT_3', 'mutations_per_gene', 'mutations_per_gene_effect', 'mutations_per_region', 'AA_TYPE_0', 'AA_TYPE_1', 'AA_TYPE_2', 'AA_TYPE_3', 'AA_TYPE_4', 'IS_PT_UKNOWN', 'is_frameshift', 'is_non_sens_mutation', 'IS_MISSENSE', 'REF_is_SNV_A', 'REF_is_SNV_C', 'REF_is_SNV_G', 'REF_is_SNV_T', 'ALT_is_SNV_A', 'ALT_is_SNV_C', 'ALT_is_SNV_G', 'ALT_is_SNV_T', 'complex_nucleotide_REF', 'complex_nucleotide_ALT', 'is_transition', 'is_transversion', 'is_indel']

In [116]:
df_significant = merge_df.drop(drop_var)

In [117]:
df_significant

ID,BM_BLAST,WBC,ANC,HB,PLT,is_normal_cyto,is_a_Female,is_down_syndrome,is_trisomy_8,is_derived_chromosome,is_lost_sex_chromosome,is_added_chromsome,is_deleted_chromsome,is_complex_karyo,CHR,VAF,DEPTH
str,f64,f64,f64,f64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64
"""KYW1""",1.0,0.329356,0.101295,0.393519,0.15894,0,0,0,0,0,0,0,0,1,2.0,0.384,0.615141
"""KYW1""",1.0,0.329356,0.101295,0.393519,0.15894,0,0,0,0,0,0,0,0,1,5.0,0.21,0.185493
"""KYW1""",1.0,0.329356,0.101295,0.393519,0.15894,0,0,0,0,0,0,0,0,1,1.0,0.027,0.446294
"""KYW1""",1.0,0.329356,0.101295,0.393519,0.15894,0,0,0,0,0,0,0,0,1,7.0,0.3639,0.196591
"""KYW2""",0.886076,0.30358,0.214197,0.615741,0.10596,0,0,0,0,1,0,0,0,1,1.0,0.072,0.653983
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""KYW1192""",0.35443,0.105012,0.037047,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,5.0,0.507,0.343242
"""KYW1192""",0.35443,0.105012,0.037047,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,5.0,0.203,0.302021
"""KYW1192""",0.35443,0.105012,0.037047,0.486111,0.089404,0,-1,-1,-1,-1,-1,-1,-1,-1,7.0,0.207,0.50654


In [119]:
pre_processed_clinical = df_significant.write_csv("../data/processed/x_test.csv")

# X_TEST

In [125]:
x_test = pd.read_csv("../data/processed/x_test.csv")

x_test = x_test.drop("is_normal_cyto" , axis=1)

In [126]:
x_test

,ID,BM_BLAST,WBC,ANC,HB,PLT,is_a_Female,is_down_syndrome,is_trisomy_8,is_derived_chromosome,is_lost_sex_chromosome,is_added_chromsome,is_deleted_chromsome,is_complex_karyo,CHR,VAF,DEPTH
0,KYW1,1.000000,0.329356,0.101295,0.393519,0.158940,0,0,0,0,0,0,0,1,2.0,0.3840,0.615141
1,KYW1,1.000000,0.329356,0.101295,0.393519,0.158940,0,0,0,0,0,0,0,1,5.0,0.2100,0.185493
2,KYW1,1.000000,0.329356,0.101295,0.393519,0.158940,0,0,0,0,0,0,0,1,1.0,0.0270,0.446294
3,KYW1,1.000000,0.329356,0.101295,0.393519,0.158940,0,0,0,0,0,0,0,1,7.0,0.3639,0.196591
4,KYW2,0.886076,0.303580,0.214197,0.615741,0.105960,0,0,0,1,0,0,0,1,1.0,0.0720,0.653983
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3223,KYW1192,0.354430,0.105012,0.037047,0.486111,0.089404,-1,-1,-1,-1,-1,-1,-1,-1,5.0,0.5070,0.343242
3224,KYW1192,0.354430,0.105012,0.037047,0.486111,0.089404,-1,-1,-1,-1,-1,-1,-1,-1,5.0,0.2030,0.302021
3225,KYW1192,0.354430,0.105012,0.037047,0.486111,0.089404,-1,-1,-1,-1,-1,-1,-1,-1,7.0,0.2070,0.506540
3226,KYW1193,0.354430,0.105012,0.037047,0.486111,0.089404,-1,-1,-1,-1,-1,-1,-1,-1,10.0,0.4200,0.304400


In [127]:
ids = x_test["ID"]
x_test_no_id = x_test.drop(columns=["ID"])

In [128]:
x_train = pd.read_csv("../data/processed/x_train.csv")

x_train = x_train.drop("Unnamed: 0" , axis=1)

In [129]:
x_train

,BM_BLAST,WBC,ANC,HB,PLT,is_a_Female,is_down_syndrome,is_trisomy_8,is_derived_chromosome,is_lost_sex_chromosome,is_added_chromsome,is_deleted_chromsome,is_complex_karyo,CHR,VAF,DEPTH,time,event
0,0.756757,0.209677,0.025890,0.291667,0.246835,0,0,0,0,0,0,0,0,21.0,0.3500,1.000000,1.115068,True
1,0.756757,0.209677,0.025890,0.291667,0.246835,0,0,0,0,0,0,0,0,2.0,0.4220,0.771723,1.115068,True
2,0.756757,0.209677,0.025890,0.291667,0.246835,0,0,0,0,0,0,0,0,13.0,0.0910,0.881969,1.115068,True
3,0.756757,0.209677,0.025890,0.291667,0.246835,0,0,0,0,0,0,0,0,13.0,0.1400,0.820955,1.115068,True
4,0.756757,0.209677,0.025890,0.291667,0.246835,0,0,0,0,0,0,0,0,16.0,0.1477,0.556701,1.115068,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11224,0.000000,0.201613,0.093204,0.347222,0.500000,0,0,0,0,0,0,0,0,7.0,0.3340,0.638754,2.290411,False
11225,0.000000,0.201613,0.093204,0.347222,0.500000,0,0,0,0,0,0,0,0,21.0,0.0230,0.591206,2.290411,False
11226,0.000000,0.201613,0.093204,0.347222,0.500000,0,0,0,0,0,0,0,0,4.0,0.3360,0.448559,2.290411,False
11227,0.000000,0.201613,0.093204,0.347222,0.500000,0,0,0,0,0,0,0,0,20.0,0.3381,0.262150,2.290411,False


In [180]:
model = GradientBoostingSurvivalAnalysis(learning_rate=1 , max_depth=5,n_estimators=200)

X_train = x_train.drop(["time" , "event"] , axis=1)

y_train = Surv.from_dataframe(event = 'event' , time='time' , data=x_train)

X_train_g = X_train.groupby("BM_BLAST").agg("mean")

X_train_g


,WBC,ANC,HB,PLT,is_a_Female,is_down_syndrome,is_trisomy_8,is_derived_chromosome,is_lost_sex_chromosome,is_added_chromsome,is_deleted_chromsome,is_complex_karyo,CHR,VAF,DEPTH
BM_BLAST,,,,,,,,,,,,,,,
0.000000,0.401225,0.352565,0.483150,0.392743,0.147503,-0.146341,-0.083624,-0.133566,-0.085947,-0.077816,-0.088269,-0.112660,10.265970,0.286828,0.452173
0.010811,0.290323,0.253251,0.428030,0.459340,0.454545,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.909091,0.292127,0.366352
0.016216,0.540323,0.750809,0.217593,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.500000,0.494800,0.092047
0.021622,0.498495,0.483944,0.499691,0.333896,0.433333,0.000000,0.166667,0.066667,0.000000,0.166667,0.000000,0.066667,6.600000,0.242257,0.439365
0.027027,0.537965,0.491655,0.435434,0.405347,0.500000,0.000000,0.173077,0.000000,0.076923,0.250000,0.038462,0.057692,11.730769,0.268670,0.465184
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0.891892,1.000000,1.000000,0.610185,0.185654,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.000000,0.501380,0.469262
0.918919,0.420788,0.278145,0.398199,0.280195,0.098214,-0.169643,-0.133929,-0.187500,-0.205357,-0.116071,-0.187500,-0.187500,12.553571,0.343432,0.388646
0.945946,0.415074,0.282798,0.562037,0.085524,0.384615,0.000000,0.384615,0.000000,0.000000,0.384615,0.000000,0.000000,12.461538,0.279968,0.461927


In [132]:
model.fit(X_train, y_train)

y_pred = model.predict(x_test_no_id)

results = pd.DataFrame({
    "ID": ids,
    "y_pred": y_pred
})

results

,ID,y_pred
0,KYW1,1.701638
1,KYW1,2.642933
2,KYW1,2.721079
3,KYW1,2.796027
4,KYW2,-1.337788
...,...,...
3223,KYW1192,1.750339
3224,KYW1192,1.464063
3225,KYW1192,1.937831
3226,KYW1193,2.184042


In [154]:
results = results.rename(columns={"y_pred" : "risk_score"})

In [162]:
result_final = results.groupby("ID").agg("max")

In [163]:
result_final

,risk_score
ID,
KYW1,2.796027
KYW10,1.045498
KYW100,5.711216
KYW1000,2.861245
KYW1001,0.781387
...,...
KYW995,0.373550
KYW996,0.512342
KYW997,-4.638495


In [ ]:
exported_res = result_final.to_csv("../data/final_res.csv")
